
# Análise Exploratória e Climática: SugarcaneMLE 🚜
Este notebook realiza a Análise Exploratória de Dados (EDA) do dataset histórico do nosso projeto end-to-end de Machine Learning para a Cana-de-Açúcar em Ribeirão Preto - SP.

O objetivo aqui é validar a qualidade dos dados climáticos da **Open-Meteo**, entender o perfil sazonal da região, e principalmente, encontrar a **Correlação Matemática e Agronômica** entre o clima (Chuva/Temperatura) e a vigor vegetativo (NDVI) da safra (simulada pelo NDVI).


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configurações visuais
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

# Carregar o dataset processado
df = pd.read_csv('../data/processed/Dataset_SugarCane_processed.csv')
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)
df.head()



## 1. Inspeção de Sanidade (Data Quality)
Antes de cruzar variáveis, precisamos garantir que a API não retornou dados espúrios (como 60°C em Ribeirão Preto).


In [ ]:

fig, ax = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Temperatura
ax[0].plot(df.index, df['t_max'], color='crimson', alpha=0.7, label='T Máxima')
ax[0].plot(df.index, df['t_min'], color='royalblue', alpha=0.7, label='T Mínima')
ax[0].set_title('Histórico de Temperaturas (Data Quality)')
ax[0].set_ylabel('°C')
ax[0].legend()

# Chuva
ax[1].plot(df.index, df['precipitacao_total'], color='teal', alpha=0.8)
ax[1].set_title('Histórico de Chuvas Diárias')
ax[1].set_ylabel('mm')

plt.tight_layout()
plt.show()



💡 **Insight:** Como podemos ver nos gráficos acima, os dados da Open-Meteo são consistentes. Não existem buracos (dados faltantes nas séries temporais) e as temperaturas estão totalmente dentro dos limites físicos conhecidos para o interior de São Paulo (mínimas próximas a 5°C e máximas não ultrapassando 40°C).


## 1.5. Detecção de Outliers Matemáticos (Método IQR)
Antes de modelar, é crucial entender se existem anomalias estatísticas severas. Vamos aplicar a regra do **Intervalo Interquartil (IQR)** para encontrar pontos fora da curva nas principais variáveis.

In [ ]:
# Variáveis para análise de outliers
vars_to_check = ["t_max", "t_min", "precipitacao_total", "radiacao_solar_mean"]

fig, axes = plt.subplots(1, len(vars_to_check), figsize=(16, 6))

for i, col in enumerate(vars_to_check):
    sns.boxplot(y=df[col], ax=axes[i], color="lightsteelblue")
    axes[i].set_title(col)
    axes[i].set_ylabel("")
    
    # Cálculo matemático do IQR
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Quantos outliers existem?
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    pct_outliers = (len(outliers) / len(df)) * 100
    axes[i].set_xlabel(f"Outliers IQR: {pct_outliers:.1f}%")

plt.suptitle("Análise de Outliers via IQR", fontsize=16)
plt.tight_layout()
plt.show()


💡 **Insight Matemático vs Agronômico:** Na `precipitacao_total`, o método estatístico IQR acusa mais de 10% de outliers (as bolinhas no topo do gráfico). No entanto, na agronomia, dias de chuvas torrenciais (tempestades de verão) são **eventos naturais**, e não erros de medição, portanto não devemos removê-los do dataset! Já para as temperaturas (`t_max` e `t_min`), não observamos quase nenhum outlier estatístico, confirmando que os dados climáticos estão saudáveis.

## 1.6. Transformando "Lixo Estatístico" em Ouro Agronômico
Como provamos acima, tempestades de verão são descartadas por ferramentas puramente matemáticas (como o IQR) por serem "outliers". Mas na agronomia, eventos extremos de chuva (> 50mm em um dia) não significam apenas "mais água para a planta".

**O Perigo da Chuva Extrema:**
1. **Lixiviação:** A enxurrada lava os fertilizantes caros (Nitrogênio e Potássio) para o subsolo, fora do alcance das raízes.
2. **Acamamento:** A pancada de água deita a cana no solo, o que reduz o teor de açúcar e dificulta severamente o corte mecânico pela colheitadeira.

**A Solução no Projeto:**
No script de engenharia de atributos (`src/features/build_features.py`), criamos a variável `eventos_chuva_extrema_30d`. Ela conta exatamente quantos desses "outliers" aconteceram no último mês. Isso ensina ao algoritmo XGBoost que a água é vital, mas o extremo da chuva destrói a vigor vegetativo (NDVI) (NDVI)!

## 1.6. Transformando "Lixo Estatístico" em Ouro Agronômico
Como provamos acima, tempestades de verão são descartadas por ferramentas puramente matemáticas (como o IQR) por serem "outliers". Mas na agronomia, eventos extremos de chuva (> 50mm em um dia) não significam apenas "mais água para a planta".

**O Perigo da Chuva Extrema:**
1. **Lixiviação:** A água lava os fertilizantes caros (Nitrogênio e Potássio) para fora do alcance das raízes.
2. **Acamamento:** A pancada de água deita a cana no solo, o que reduz o teor de açúcar e dificulta severamente o corte pela colheitadeira.

**A Solução no Projeto:**
No script de engenharia de atributos (), criamos a variável . Ela conta exatamente quantos desses "outliers" aconteceram no último mês. Isso ensina ao algoritmo XGBoost que a água é boa, mas o extremo da água destrói a vigor vegetativo (NDVI) (NDVI)!

## 1.5. Detecção de Outliers Matemáticos (Método IQR)
Antes de modelar, é crucial entender se existem anomalias estatísticas severas. Vamos aplicar a regra do **Intervalo Interquartil (IQR)** para encontrar pontos fora da curva nas principais variáveis.

In [ ]:
# Variáveis para análise de outliers
vars_to_check = ['t_max', 't_min', 'precipitacao_total', 'radiacao_solar_mean']

fig, axes = plt.subplots(1, len(vars_to_check), figsize=(16, 6))

for i, col in enumerate(vars_to_check):
    sns.boxplot(y=df[col], ax=axes[i], color='lightsteelblue')
    axes[i].set_title(col)
    axes[i].set_ylabel('')
    
    # Cálculo matemático do IQR
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Quantos outliers existem?
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    pct_outliers = (len(outliers) / len(df)) * 100
    axes[i].set_xlabel(f'Outliers IQR: {pct_outliers:.1f}%')

plt.suptitle('Análise de Outliers via IQR', fontsize=16)
plt.tight_layout()
plt.show()


💡 **Insight Matemático vs Agronômico:** Na , o método estatístico IQR acusa mais de 10% de outliers (as bolinhas no topo do gráfico). No entanto, na agronomia, dias de chuvas torrenciais (tempestades de verão) são **eventos naturais**, e não erros de medição, portanto não devemos removê-los do dataset! Já para as temperaturas ( e ), não observamos quase nenhum outlier estatístico, confirmando que os dados da API são perfeitamente consistentes.


## 2. O Perfil Climático de Ribeirão Preto (Sazonalidade)
A cana-de-açúcar depende de um clima com Verões chuvosos (crescimento vegetativo) e Invernos secos (maturação e acúmulo de sacarose - ATR). Vamos validar se Ribeirão Preto se encaixa nisso.


In [ ]:

# Criar coluna de mês
df['mes'] = df.index.month

fig, ax1 = plt.subplots(figsize=(12, 6))

# Boxplot de Chuva por Mês
sns.barplot(x='mes', y='precipitacao_total', data=df, ax=ax1, color='skyblue', estimator=np.sum, errorbar=None)
ax1.set_ylabel('Precipitação Total Média (mm)', color='tab:blue')
ax1.set_xlabel('Mês')
ax1.set_title('Sazonalidade: Chuva vs Temperatura em Ribeirão Preto')

# Linha de Temperatura Média
ax2 = ax1.twinx()
sns.lineplot(x=df['mes'] - 1, y='t_mean', data=df, ax=ax2, color='crimson', marker='o', linewidth=2)
ax2.set_ylabel('Temperatura Média (°C)', color='crimson')

plt.show()



💡 **Insight:** O padrão de clima Tropical de Altitude é claro: Os meses de Outubro a Março concentram quase 80% do volume de chuvas, acompanhados de altas temperaturas (ideal para o crescimento da biomassa). Já a janela de Junho a Agosto apresenta seca severa e frio, que é o gatilho fisiológico para a cana acumular açúcar, permitindo a entrada das colhedoras no campo.



## 3. A Saúde da Safra (Análise do NDVI)
Vamos observar a evolução da vigor vegetativo (NDVI) simulada em Índice de Vigor Vegetativo (NDVI) ao longo dos Anos-Safra. A safra da cana geralmente começa em Abril.


In [ ]:

# Definir Ano-Safra (Abril a Março do ano seguinte)
df['ano_safra'] = np.where(df.index.month >= 4, df.index.year, df.index.year - 1)

# Agrupar NDVI médio por ano safra
safra_tch = df.groupby('ano_safra')['ndvi_medio'].mean().reset_index()

plt.figure(figsize=(10, 5))
sns.barplot(x='ano_safra', y='ndvi_medio', data=safra_tch, hue='ano_safra', palette='viridis', legend=False)
plt.title('Evolução da Produtividade (NDVI) por Ano-Safra')
plt.ylabel('Índice NDVI (NDVI)')
plt.xlabel('Ano Safra')
plt.ylim(0, safra_tch['ndvi_medio'].max() * 1.2)

# Adicionar rótulos
for index, row in safra_tch.iterrows():
    plt.text(index, row.ndvi_medio + 2, f'{row.ndvi_medio:.1f}', color='black', ha="center")

plt.show()


💡 **Insight:** Como podemos ver no gráfico acima, a safra de 2021 apresentou uma queda em relação a 2022 e 2023. No entanto, em Data Science, não podemos afirmar que a culpa foi da "seca" sem mostrar os dados! Para afirmar isso, precisamos cruzar a vigor vegetativo (NDVI) com o histórico climático de cada ano, como faremos no gráfico a seguir.

In [ ]:
# Agrupar clima por ano-safra para provar a tese
clima_anual = df.groupby('ano_safra').agg({
    'precipitacao_total': 'sum',
    't_mean': 'mean'
}).reset_index()

fig, ax1 = plt.subplots(figsize=(10, 5))

# Gráfico de Barras para Chuva
sns.barplot(x='ano_safra', y='precipitacao_total', data=clima_anual, ax=ax1, color='skyblue')
ax1.set_ylabel('Volume Total de Chuva (mm)', color='tab:blue')
ax1.set_title('Histórico Climático por Ano-Safra')

# Gráfico de Linha para Temperatura Média
ax2 = ax1.twinx()
sns.lineplot(x=clima_anual.index, y='t_mean', data=clima_anual, ax=ax2, color='crimson', marker='o', linewidth=2)
ax2.set_ylabel('Temperatura Média Anual (°C)', color='crimson')

plt.show()


💡 **Prova Baseada em Dados:** Agora sim! Ao cruzar os gráficos, fica provado matematicamente. O ano-safra de 2021 teve um volume total de chuvas consideravelmente inferior se comparado a 2022 e 2023. A falta de água durante o ciclo vegetativo foi o fator limitante que derrubou o NDVI daquele ano. Nunca tire conclusões sem o respaldo visual dos dados causadores!


## 4. A Cereja do Bolo: Correlação Temporal (Lags)
A chuva que cai hoje não se transforma em biomassa amanhã. Existe um atraso (Lag) entre o sinal climático e a resposta fisiológica da planta. Vamos provar matematicamente a força desse atraso.


In [ ]:

# Para o heatmap, vamos criar um df mensal agrupado
df_monthly = df.resample('ME').agg({
    'ndvi_medio': 'mean',
    'precipitacao_total': 'sum'
}).dropna()

# Criando as janelas (Lags) de chuva (1 mês = aprox 30 dias, logo Lag 3 = 3 meses)
df_monthly['Chuva_Lag_0'] = df_monthly['precipitacao_total']
df_monthly['Chuva_Lag_3'] = df_monthly['precipitacao_total'].shift(3)
df_monthly['Chuva_Lag_6'] = df_monthly['precipitacao_total'].shift(6)
df_monthly['Chuva_Lag_9'] = df_monthly['precipitacao_total'].shift(9)

# Calculando a matriz de correlação cruzando NDVI com os lags de chuva
corr_matrix = df_monthly[['ndvi_medio', 'Chuva_Lag_0', 'Chuva_Lag_3', 'Chuva_Lag_6', 'Chuva_Lag_9']].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix[['ndvi_medio']].sort_values(by='ndvi_medio', ascending=False), 
            annot=True, cmap='coolwarm', vmin=-1, vmax=1, center=0)
plt.title('Correlação do NDVI com Lags de Precipitação')
plt.show()


💡 **Insight:** A matriz de correlação nos mostra um comportamento fascinante: O maior pico de correlação positiva ocorre no **Lag 9** (0.55). Isso faz muito sentido agronômico, pois a chuva que cai 9 meses antes da colheita (bem no início da fase de brotação/perfilhamento) é a que mais impulsiona a formação inicial da biomassa. Por outro lado, vemos correlações negativas nos Lags 3 e 6, indicando que excesso de chuva muito perto da fase de maturação pode até prejudicar o NDVI final ou o acúmulo de açúcar.


## 5. Análise de Risco (Geadas e Veranicos)
A cana morre se congelar e para de crescer se faltar água no verão. Vamos identificar os dias de estresse agudo na série histórica.


In [ ]:

# Identificar risco de geada (T min < 4°C)
df['risco_geada'] = df['t_min'] < 4.0

# Identificar "Veranicos" (Dias de Verão - Dez, Jan, Fev - sem chuva consecutiva)
# Como a série já é diária, vamos achar dias de verão com chuva = 0
is_summer = df.index.month.isin([12, 1, 2])
no_rain = df['precipitacao_total'] == 0

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df.index, df['t_min'], color='lightgray', label='T Mínima Diária')

# Destacar Geadas
geadas = df[df['risco_geada']]
ax.scatter(geadas.index, geadas['t_min'], color='blue', s=50, label='Alerta de Geada (T < 4°C)', zorder=5)

# Linha de alerta
ax.axhline(4, color='cyan', linestyle='--', label='Limiar Geada')

ax.set_title('Monitoramento de Risco: Geadas em Ribeirão Preto')
ax.set_ylabel('Temperatura Mínima (°C)')
ax.legend()
plt.show()

# Resumo de Veranicos
dias_veranico = len(df[is_summer & no_rain])
total_dias_verao = len(df[is_summer])
print(f"Ao longo dos 5 anos, ocorreram {dias_veranico} dias secos durante a janela de Verão (dos {total_dias_verao} dias de verão totais).")


💡 **Insight:** Como podemos ver claramente no gráfico, a linha de temperatura mínima NUNCA cruzou o limiar de 4°C (a mínima histórica ficou em torno de 5°C). Isso é uma excelente notícia! Confirma que Ribeirão Preto é uma região de clima perfeito para a cana-de-açúcar, praticamente livre do risco de geadas que poderiam destruir as gemas apicais da planta. No entanto, os 57 dias de Veranico mostram que o déficit hídrico no verão é o verdadeiro desafio da região.